# CSV Data Ingestion System — Architecture

**Architecture flow:**

1. **Catalog** → `workspace`
2. **Schema** → select existing OR create new
   * New schema → Volume `raw_files` created **automatically**
   * Existing schema → list volumes → user selects one OR creates a NEW volume with a custom name (option always shown, even if none exist)
3. **Volume** → raw CSV files stored here
4. **Table** → create + define schema
5. **Data ingestion** → CSV (from Volume) **OR** manual entry
6. **Spark DataFrame** → loaded into the Delta table

In [0]:
%sql
SHOW CATALOGS


In [0]:
%sql
SHOW SCHEMAS IN workspace;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.csv_ingestion;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.csv_ingestion.raw_files;

In [0]:
%sql
SHOW VOLUMES IN workspace.csv_ingestion;

In [0]:
catalog_name = "workspace"

schemas = spark.sql(f"SHOW SCHEMAS IN {catalog_name}")

schemas.show()

In [0]:
catalog_name = "workspace"

print("""
========================================
       CSV DATA INGESTION SYSTEM
========================================
STEP 1 — Select or Create SCHEMA

1. Use existing schema
2. Create new schema

========================================
""")

choice = input("Enter your choice (1 or 2): ")

selected_schema = None
selected_volume = None

if choice == "1":

    # ---------- CASE 1: EXISTING SCHEMA ----------

    print("\nAvailable schemas:\n")

    schemas_df = spark.sql(f"SHOW SCHEMAS IN {catalog_name}")

    schemas = [
        row.databaseName
        for row in schemas_df.collect()
        if row.databaseName != "information_schema"
    ]

    for i, schema in enumerate(schemas, start=1):
        print(f"{i}. {schema}")

    schema_choice = input("\nEnter schema number: ")

    schema_index = int(schema_choice) - 1

    selected_schema = schemas[schema_index]

    print("\nYou selected schema:", selected_schema)

    # ---------- VOLUME LOGIC (existing schema) ----------

    volumes_df = spark.sql(
        f"SHOW VOLUMES IN {catalog_name}.{selected_schema}"
    )

    volumes = [row.volume_name for row in volumes_df.collect()]

    if len(volumes) > 0:

        print(f"\nVolumes in schema '{selected_schema}':\n")

        for i, v in enumerate(volumes, start=1):
            print(f"{i}. {v}")

        new_volume_option = len(volumes) + 1

        print(f"{new_volume_option}. Create a NEW volume")

        volume_choice = input("\nEnter volume number: ")

        volume_index = int(volume_choice) - 1

        if volume_index == len(volumes):

            # ---------- CREATE NEW VOLUME (user-defined name) ----------

            new_volume = input("Enter new volume name: ")

            spark.sql(
                f"CREATE VOLUME IF NOT EXISTS "
                f"{catalog_name}.{selected_schema}.{new_volume}"
            )

            selected_volume = new_volume

            print(f"\nVolume '{new_volume}' created successfully.")

        else:

            selected_volume = volumes[volume_index]

            print("\nYou selected volume:", selected_volume)

    else:

        print(f"\nNo volumes found in schema '{selected_schema}'.")

        create_volume = input(
            "Create a new volume? (yes/no): "
        ).strip().lower()

        if create_volume == "yes":

            new_volume = input("Enter new volume name: ")

            spark.sql(
                f"CREATE VOLUME IF NOT EXISTS "
                f"{catalog_name}.{selected_schema}.{new_volume}"
            )

            selected_volume = new_volume

            print(f"\nVolume '{new_volume}' created successfully.")

        else:

            print("\nNo volume selected — CSV storage unavailable.")

elif choice == "2":

    # ---------- CASE 2: NEW SCHEMA ----------

    new_schema = input("\nEnter new schema name: ")

    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{new_schema}"
    )

    selected_schema = new_schema

    print(f"\nSchema '{new_schema}' created successfully.")

    # ---------- VOLUME LOGIC (new schema → automatic) ----------

    auto_volume = "raw_files"

    spark.sql(
        f"CREATE VOLUME IF NOT EXISTS "
        f"{catalog_name}.{selected_schema}.{auto_volume}"
    )

    selected_volume = auto_volume

    print(f"Volume '{auto_volume}' created automatically.")
    print("(New schemas always get a dedicated raw-file volume.)")

else:

    print("\nInvalid choice. Please enter 1 or 2.")


# ---------- FINAL SELECTION SUMMARY ----------

print("\n" + "=" * 40)
print("SELECTION COMPLETE")
print("=" * 40)
print(f"Catalog : {catalog_name}")
print(f"Schema  : {selected_schema}")

if selected_volume:

    volume_path = (
        f"/Volumes/{catalog_name}/{selected_schema}/{selected_volume}"
    )

    print(f"Volume  : {selected_volume}")
    print(f"Path    : {volume_path}")

print("=" * 40)

In [0]:
%sql
SHOW VOLUMES IN workspace.retail;

In [0]:
table_name = input("\nEnter table name: ")

print("\nSelected schema:", selected_schema)
print("Table name:", table_name)

In [0]:
num_columns = int(input("\nHow many columns do you want? "))

print(f"\nYou selected {num_columns} columns.")

In [0]:
columns = []

data_types = {
    "1": "INT",
    "2": "BIGINT",
    "3": "DOUBLE",
    "4": "STRING",
    "5": "BOOLEAN",
    "6": "DATE",
    "7": "TIMESTAMP"
}

for i in range(num_columns):

    print(f"\n--- Column {i + 1} ---")

    column_name = input("Enter column name: ")

    print("""
Choose data type:

1. INT        → Whole numbers
2. BIGINT     → Large whole numbers
3. DOUBLE     → Decimal numbers
4. STRING     → Text
5. BOOLEAN    → True / False
6. DATE       → YYYY-MM-DD
7. TIMESTAMP  → Date + Time
""")

    type_choice = input("Enter data type number (1-7): ")

    column_type = data_types.get(type_choice)

    if column_type is None:
        print("Invalid data type. Please choose 1-7.")
        break

    columns.append((column_name, column_type))


print("\nYour table structure:")

for column_name, column_type in columns:
    print(f"{column_name} → {column_type}")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    LongType,
    DoubleType,
    StringType,
    BooleanType,
    DateType,
    TimestampType
)

In [0]:
spark_types = {
    "INT": IntegerType(),
    "BIGINT": LongType(),
    "DOUBLE": DoubleType(),
    "STRING": StringType(),
    "BOOLEAN": BooleanType(),
    "DATE": DateType(),
    "TIMESTAMP": TimestampType()
}

In [0]:
fields = []

for column_name, column_type in columns:

    spark_data_type = spark_types[column_type]

    field = StructField(
        column_name,
        spark_data_type,
        True
    )

    fields.append(field)

In [0]:
schema = StructType(fields)

In [0]:
print("\nGenerated Spark Schema:")
schema.simpleString()

In [0]:
print(schema.treeString())

In [0]:
full_table_name = f"{catalog_name}.{selected_schema}.{table_name}"

print("Table to be created:", full_table_name)

In [0]:
column_definitions = ", ".join(
    f"{field.name} {field.dataType.simpleString().upper()}"
    for field in schema.fields
)

print(column_definitions)

In [0]:
# Explanation of the created Spark Schema

type_explanations = {
    "INT": "whole numbers (32-bit)",
    "BIGINT": "large whole numbers (64-bit)",
    "DOUBLE": "decimal (floating-point) numbers",
    "STRING": "text values",
    "BOOLEAN": "true / false values",
    "DATE": "calendar dates (YYYY-MM-DD)",
    "TIMESTAMP": "date + time values"
}

print(f"The created schema has {len(schema.fields)} columns:\n")

for field in schema.fields:

    type_name = field.dataType.simpleString().upper()

    meaning = type_explanations.get(type_name, "custom type")

    if field.nullable:
        null_rule = "NULLs are ALLOWED"
    else:
        null_rule = "NULLs are NOT allowed"

    print(f"• {field.name} ({type_name}) → {meaning}; {null_rule}")

In [0]:
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {full_table_name} (
    {column_definitions}
)
USING DELTA
"""

print(create_table_sql)

In [0]:
spark.sql(create_table_sql)

print(f"\nTable '{full_table_name}' created successfully!")

In [0]:
spark.sql(f"DESCRIBE {full_table_name}").show()

In [0]:
print("""
========================================
          DATA INPUT METHOD
========================================

1. Upload CSV file → stored in selected Volume
2. Enter data manually

========================================
""")

data_choice = input("Enter your choice (1 or 2): ")

if data_choice == "1":

    print("\nYou selected: Upload CSV file")

    if selected_volume:
        print("Raw CSV will be stored in:", volume_path)
    else:
        print("WARNING: No volume selected — CSV upload unavailable.")

elif data_choice == "2":

    print("\nYou selected: Enter data manually")

    num_records = int(input("\nHow many records do you want to enter? "))

    print(f"\nYou will enter {num_records} records.")

else:

    print("\nInvalid choice. Please enter 1 or 2.")

In [0]:
# ============ BRANCH 1: CSV UPLOAD ============
# Store the raw CSV file into the selected Volume

if data_choice == "1" and selected_volume:

    csv_file_name = input("\nEnter CSV file name (e.g., products.csv): ")

    if not csv_file_name.endswith(".csv"):
        csv_file_name += ".csv"

    csv_volume_path = f"{volume_path}/{csv_file_name}"

    print("\nEnter CSV rows — first row must be the header.")
    print("Type 'done' on an empty prompt to finish.\n")

    csv_lines = []

    while True:
        line = input("> ")
        if line.strip().lower() == "done":
            break
        csv_lines.append(line)

    csv_content = "\n".join(csv_lines) + "\n"

    dbutils.fs.put(csv_volume_path, csv_content, overwrite=True)

    print("\nRaw CSV file stored in Volume:")
    print(csv_volume_path)

elif data_choice == "1":

    print("No volume selected — run the schema selection cell first.")

In [0]:
# ============ BRANCH 1: READ CSV FROM VOLUME ============

if data_choice == "1" and selected_volume:

    csv_df = spark.read \
        .option("header", True) \
        .schema(schema) \
        .csv(csv_volume_path)

    print("CSV data read from Volume:\n")

    csv_df.show()

In [0]:
def convert_value(value, column_type):

    if value == "":
        return None

    if column_type == "INT":
        return int(value)

    elif column_type == "BIGINT":
        return int(value)

    elif column_type == "DOUBLE":
        return float(value)

    elif column_type == "STRING":
        return value

    elif column_type == "BOOLEAN":

        value = value.strip().lower()

        if value == "true":
            return True

        elif value == "false":
            return False

        else:
            raise ValueError("Please enter True or False")

    elif column_type == "DATE":

        from datetime import datetime

        return datetime.strptime(
            value,
            "%Y-%m-%d"
        ).date()

    elif column_type == "TIMESTAMP":

        return datetime.now()

In [0]:
from datetime import datetime

# ============ BRANCH 2: MANUAL DATA ENTRY ============

if data_choice == "2":

    data = []

    for record_num in range(num_records):

        print(f"\n--- Record {record_num + 1} ---")

        row = []

        for column_name, column_type in columns:

            if column_type == "TIMESTAMP":

                value = datetime.now()

                print(f"{column_name} → {value}")

            else:

                value = input(
                    f"Enter {column_name} ({column_type}): "
                )

            row.append(value)

        data.append(row)

    print("\nData entered successfully!")
    print(data)

In [0]:
if data_choice == "2":
    print(data)

In [0]:
# ============ BRANCH 2: MANUAL DATA → DATAFRAME ============

if data_choice == "2":

    converted_data = []

    for row in data:
        converted_row = []
        for i, (value, (col_name, col_type)) in enumerate(zip(row, columns)):
            if value == "":
                converted_row.append(None)
            elif col_type == "INT":
                converted_row.append(int(value))
            elif col_type == "BIGINT":
                converted_row.append(int(value))
            elif col_type == "DOUBLE":
                converted_row.append(float(value))
            elif col_type == "STRING":
                converted_row.append(value)
            elif col_type == "BOOLEAN":
                value = value.strip().lower()
                if value == "true":
                    converted_row.append(True)
                elif value == "false":
                    converted_row.append(False)
                else:
                    raise ValueError("Please enter True or False")
            elif col_type == "DATE":
                from datetime import datetime
                converted_row.append(datetime.strptime(value, "%Y-%m-%d").date())
            elif col_type == "TIMESTAMP":
                converted_row.append(value)
        converted_data.append(converted_row)

    manual_df = spark.createDataFrame(
        converted_data,
        schema=schema
    )

    print("\nManual data as Spark DataFrame:\n")

    manual_df.show()

    print("DataFrame schema:")

    manual_df.printSchema()

In [0]:
# ============ CONVERGENCE: FINAL INGESTION DATAFRAME ============
# Both branches produce a Spark DataFrame → one Delta table

if data_choice == "1":
    ingest_df = csv_df
    print("Ingestion source : CSV file from Volume")
elif data_choice == "2":
    ingest_df = manual_df
    print("Ingestion source : Manual entry")

print("\nFinal DataFrame to load into the Delta table:\n")

ingest_df.show()

In [0]:
# ============ LOAD DATAFRAME INTO DELTA TABLE ============

# Both branches converge here: ingest_df (CSV or manual) → Delta table

ingest_df.write \
    .mode("append") \
    .saveAsTable(full_table_name)

print(f"\nData successfully inserted into {full_table_name}")

print("\nIngestion summary:")
print("  Catalog :", catalog_name)
print("  Schema  :", selected_schema)
if selected_volume:
    print("  Volume  :", selected_volume)
print("  Table   :", full_table_name)

In [0]:
spark.sql(
    f"SELECT * FROM {full_table_name}"
).show()

In [0]:
# ============ EXPORT: DELTA TABLE → CSV FILE IN VOLUME ============

# Read the Delta table as a DataFrame
export_df = spark.sql(f"SELECT * FROM {full_table_name}")

# coalesce(1) → write all rows into ONE single part file
export_df = export_df.coalesce(1)

# Temporary export directory inside the selected Volume
export_dir = f"{volume_path}/{table_name}_csv_export"

export_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(export_dir)

# Locate the generated part file inside the directory
part_file = [
    f.path for f in dbutils.fs.ls(export_dir)
    if f.name.startswith("part-")
][0]

# Save it as a proper single .csv file in the Volume
csv_file_path = f"{volume_path}/{table_name}.csv"

dbutils.fs.cp(part_file, csv_file_path)

# Clean up the temporary export directory
dbutils.fs.rm(export_dir, True)

print("Table exported as CSV file:")
print(csv_file_path)

In [0]:
# ============ VERIFY: CSV FILE IN VOLUME ============

print("Files in the selected Volume:\n")

for f in dbutils.fs.ls(volume_path):
    print(f"• {f.name} ({f.size} bytes)")

print("\nCSV content preview:\n")

print(dbutils.fs.head(csv_file_path))

## Can we create a CODE FILE or NOTEBOOK inside the Volume?

**Yes — files CAN be stored in the Volume** (any file type — including `.py` code files and `.ipynb` notebook files).

**But** the notebook editor can only open notebooks stored in `/Workspace` — a file inside a Volume **cannot be opened directly for editing**.

| What you want | How to do it |
| --- | --- |
| Store code in the Volume | Write a `.py` file with `dbutils.fs.put()` |
| Run code stored in the Volume | `exec(dbutils.fs.head(path))` from any notebook, or `%run "/Volumes/..."` (path typed literally) |
| Store a notebook in the Volume | Copy the `.ipynb` file into the Volume (archive copy) |
| Edit a notebook stored in the Volume | Import it back into `/Workspace` first |

**Pattern:** the Volume stores the **code file + data files together**, and any notebook can run that code — so each volume is self-contained.

The next cell asks which file to create: **`.py`** (runnable code) or **`.ipynb`** (notebook archive copy).

In [0]:
# ============ STEP 6: CREATE A CODE / NOTEBOOK FILE IN THE VOLUME ============

print(f"""
========================================
     CREATE A FILE IN THE VOLUME
========================================

Selected Volume: {volume_path}

1. Create a Python code file (.py)
   → code stored in the Volume — runnable
     from any notebook (%run / exec)

2. Create a Notebook file (.ipynb)
   → archive copy stored in the Volume —
     import back into /Workspace to edit

========================================
""")

file_choice = input("Enter your choice (1 or 2): ")

if file_choice == "1":

    print("\nYou selected: Python code file (.py)")

elif file_choice == "2":

    print("\nYou selected: Notebook file (.ipynb)")

else:

    print("\nInvalid choice. Please enter 1 or 2.")

In [0]:
# ============ BRANCH 1: CREATE .py CODE FILE IN VOLUME ============

if file_choice == "1":

    code_file_path = f"{volume_path}/volume_analysis.py"

    code_content = f"""# This code file lives INSIDE the Volume
# It reads the CSV that was exported to the same Volume

csv_path = "{csv_file_path}"

print("Executing code stored in Volume file: volume_analysis.py")

volume_csv_df = spark.read \\
    .option("header", True) \\
    .option("inferSchema", True) \\
    .csv(csv_path)

volume_csv_df.show()
"""

    dbutils.fs.put(code_file_path, code_content, overwrite=True)

    print("Code file created inside the Volume:")
    print(code_file_path)

In [0]:
# ============ BRANCH 2: CREATE .ipynb NOTEBOOK FILE IN VOLUME ============

import json

if file_choice == "2":

    notebook_file_path = (
        f"{volume_path}/{selected_schema}_volume_notebook.ipynb"
    )

    notebook_json = json.dumps(
        {
            "cells": [
                {
                    "cell_type": "markdown",
                    "metadata": {},
                    "source": [
                        f"# Volume Notebook — {catalog_name}.{selected_schema}.{selected_volume}\n",
                        "\n",
                        "This notebook file is stored INSIDE the Volume.\n",
                        "Import it back into /Workspace to open and edit it.\n"
                    ]
                },
                {
                    "cell_type": "code",
                    "execution_count": None,
                    "metadata": {},
                    "outputs": [],
                    "source": [
                        "# Load the CSV stored in this Volume\n",
                        "volume_csv_df = spark.read \\\n",
                        "    .option(\"header\", True) \\\n",
                        "    .option(\"inferSchema\", True) \\\n",
                        f"    .csv(\"{csv_file_path}\")\n",
                        "\n",
                        "volume_csv_df.show()"
                    ]
                }
            ],
            "metadata": {
                "language_info": {"name": "python"}
            },
            "nbformat": 4,
            "nbformat_minor": 5
        },
        indent=1
    )

    dbutils.fs.put(notebook_file_path, notebook_json, overwrite=True)

    print("Notebook file created inside the Volume:")
    print(notebook_file_path)
    print("\nTo edit it later: import this .ipynb back into /Workspace")
    print("(Workspace sidebar → Import → choose this file from the Volume)")

In [0]:
# ============ RUN / VERIFY THE FILE CREATED IN THE VOLUME ============

if file_choice == "1":

    # Read the .py code file and execute it
    code_text = dbutils.fs.head(code_file_path)

    print("Running the .py code file from the Volume:\n")

    exec(code_text)

    # Alternative — the %run magic (must be typed literally, own cell):
    #   %run "/Volumes/workspace/retail/products/volume_analysis.py"

elif file_choice == "2":

    print("Files now in the selected Volume:\n")

    for f in dbutils.fs.ls(volume_path):
        print(f"• {f.name} ({f.size} bytes)")

    print("\nNotebook JSON preview:\n")

    print(dbutils.fs.head(notebook_file_path))

In [0]:
# ============ RUN THE .ipynb STORED IN THE VOLUME (NO IMPORT NEEDED) ============

import json

nb_file = f"{selected_schema}_volume_notebook.ipynb"

volume_files = [f.name for f in dbutils.fs.ls(volume_path)]

if nb_file in volume_files:

    nb_path = f"{volume_path}/{nb_file}"

    # Read the notebook JSON from the Volume
    notebook_json = json.loads(dbutils.fs.head(nb_path))

    # Collect all CODE cells inside the notebook
    code_cells = [
        "".join(cell["source"])
        for cell in notebook_json["cells"]
        if cell["cell_type"] == "code"
    ]

    print(f"Running {len(code_cells)} code cell(s) from: {nb_path}\n")

    for i, cell_code in enumerate(code_cells, 1):

        print(f"--- code cell {i} ---\n")

        exec(cell_code)

else:

    print(f"{nb_file} not found in this Volume.")
    print("Run STEP 6 with choice 2 first to create it.")